In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Unitäre Operationen synthetisieren

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
```
</details>
Eine unitäre Operation beschreibt eine normerhaltende Änderung an einem Quantensystem.
Für $n$ Qubits wird diese Änderung durch eine $2^n \times 2^n$ dimensionale, komplexe Matrix $U$ beschrieben, deren Adjungierte der Inversen entspricht, d. h. $U^\dagger U = \mathbb{1}$.

Das Synthetisieren spezifischer unitärer Operationen in eine Menge von Quantengattern ist eine grundlegende Aufgabe, die zum Beispiel beim Entwurf und der Anwendung von Quantenalgorithmen oder beim Kompilieren von Quantenschaltkreisen verwendet wird.

Während eine effiziente Synthese für bestimmte Klassen von Unitären möglich ist – etwa solche, die aus Clifford-Gattern aufgebaut sind oder eine Tensorproduktstruktur besitzen – fallen die meisten Unitären nicht in diese Kategorien.
Für allgemeine unitäre Matrizen ist die Synthese eine komplexe Aufgabe, deren Rechenaufwand exponentiell mit der Anzahl der Qubits wächst.
Wenn du daher eine effiziente Zerlegung für die zu implementierende Unitäre kennst, wird diese wahrscheinlich besser als eine allgemeine Synthese sein.

> **Note:** Falls keine Zerlegung verfügbar ist, stellt das Qiskit SDK dir die Werkzeuge zur Verfügung, um eine zu finden.
>     Beachte jedoch, dass dabei im Allgemeinen tiefe Schaltkreise entstehen, die sich möglicherweise nicht für den Betrieb auf verrauschten Quantencomputern eignen.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## Re-Synthese zur Schaltkreisoptimierung
Manchmal ist es sinnvoll, eine lange Folge von Ein- und Zwei-Qubit-Gattern neu zu synthetisieren, wenn sich die Länge dadurch reduzieren lässt. Der folgende Schaltkreis verwendet beispielsweise drei Zwei-Qubit-Gatter.

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Ausgabe der vorherigen Codezelle](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

Nach der erneuten Synthese mit dem folgenden Code wird jedoch nur noch ein einziges CX-Gatter benötigt. (Hier verwenden wir die Methode `QuantumCircuit.decompose()`, um die bei der Re-Synthese der Unitären verwendeten Gatter besser zu visualisieren.)